In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES
# TO THE CORRECT LOCATION (/kaggle/input) IN YOUR NOTEBOOK,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

import os
import sys
from tempfile import NamedTemporaryFile
from urllib.request import urlopen
from urllib.parse import unquote, urlparse
from urllib.error import HTTPError
from zipfile import ZipFile
import tarfile
import shutil

CHUNK_SIZE = 40960
DATA_SOURCE_MAPPING = 'plant-disease-classification-merged-dataset:https%3A%2F%2Fstorage.googleapis.com%2Fkaggle-data-sets%2F2781321%2F4803640%2Fbundle%2Farchive.zip%3FX-Goog-Algorithm%3DGOOG4-RSA-SHA256%26X-Goog-Credential%3Dgcp-kaggle-com%2540kaggle-161607.iam.gserviceaccount.com%252F20240504%252Fauto%252Fstorage%252Fgoog4_request%26X-Goog-Date%3D20240504T135432Z%26X-Goog-Expires%3D259200%26X-Goog-SignedHeaders%3Dhost%26X-Goog-Signature%3D44e6c5d951fc7cdc5b4b10f8feec28b6668c972cf9e209e3e2f6afa668bbdc95155379b8f93128cd7b827be37826c60e19814f5f4586e87881d37eb396176a77008357b4e41de1f41b41bb7cb116a21a78de18d9787b95466b66e6d1479299a51f03c3984278f019e7c995644f310d7fc29f56cbae0494d1d477ea5918516581c33e87691561d6c8f96fe40071aa5c24a00c0cde034d2ed9beaa30c04ba72f243b84cebeb3dcae9fb1ef9509329869b25c96bb44902536daf277751fea41e0c4f083191ceafaeee626641359e168019da4427bf65b354aab85fa8ee4f8d75b0264d77d7d3c08c4d4a485f9a35d84c5d5ce34bf64d8927c8c13a3a2538ccfe6f1'

KAGGLE_INPUT_PATH='/kaggle/input'
KAGGLE_WORKING_PATH='/kaggle/working'
KAGGLE_SYMLINK='kaggle'

!umount /kaggle/input/ 2> /dev/null
shutil.rmtree('/kaggle/input', ignore_errors=True)
os.makedirs(KAGGLE_INPUT_PATH, 0o777, exist_ok=True)
os.makedirs(KAGGLE_WORKING_PATH, 0o777, exist_ok=True)

try:
  os.symlink(KAGGLE_INPUT_PATH, os.path.join("..", 'input'), target_is_directory=True)
except FileExistsError:
  pass
try:
  os.symlink(KAGGLE_WORKING_PATH, os.path.join("..", 'working'), target_is_directory=True)
except FileExistsError:
  pass

for data_source_mapping in DATA_SOURCE_MAPPING.split(','):
    directory, download_url_encoded = data_source_mapping.split(':')
    download_url = unquote(download_url_encoded)
    filename = urlparse(download_url).path
    destination_path = os.path.join(KAGGLE_INPUT_PATH, directory)
    try:
        with urlopen(download_url) as fileres, NamedTemporaryFile() as tfile:
            total_length = fileres.headers['content-length']
            print(f'Downloading {directory}, {total_length} bytes compressed')
            dl = 0
            data = fileres.read(CHUNK_SIZE)
            while len(data) > 0:
                dl += len(data)
                tfile.write(data)
                done = int(50 * dl / int(total_length))
                sys.stdout.write(f"\r[{'=' * done}{' ' * (50-done)}] {dl} bytes downloaded")
                sys.stdout.flush()
                data = fileres.read(CHUNK_SIZE)
            if filename.endswith('.zip'):
              with ZipFile(tfile) as zfile:
                zfile.extractall(destination_path)
            else:
              with tarfile.open(tfile.name) as tarfile:
                tarfile.extractall(destination_path)
            print(f'\nDownloaded and uncompressed: {directory}')
    except HTTPError as e:
        print(f'Failed to load (likely expired) {download_url} to path {destination_path}')
        continue
    except OSError as e:
        print(f'Failed to load {download_url} to path {destination_path}')
        continue

print('Data source import complete.')


[==================================================] 18611874922 bytes downloaded
Downloaded and uncompressed: plant-disease-classification-merged-dataset
Data source import complete.


In [ ]:
import os
import shutil

# List of folder prefixes to be deleted
folders_to_delete = ['Cassava', 'Corn', 'Grape', 'Jamun', 'Rice', 'Soybean', 'Sugarcane', 'Wheat']

# Get the current directory
current_directory = "/kaggle/input/plant-disease-classification-merged-dataset"

# Function to recursively delete folders and their contents
def delete_folders(folder_prefix):
    for root, dirs, files in os.walk(current_directory):
        for folder in dirs:
            if folder.startswith(folder_prefix):
                folder_path = os.path.join(root, folder)

                # Display the folder to be deleted
                print(f"Deleting: {folder_path}")
                shutil.rmtree(folder_path)
                print(f"Folder '{folder}' and its contents deleted.")

# Iterate through each folder prefix and delete corresponding folders
for prefix in folders_to_delete:
    delete_folders(prefix)


Deleting: /kaggle/input/plant-disease-classification-merged-dataset/Cassava__green_mottle
Folder 'Cassava__green_mottle' and its contents deleted.
Deleting: /kaggle/input/plant-disease-classification-merged-dataset/Cassava__healthy
Folder 'Cassava__healthy' and its contents deleted.
Deleting: /kaggle/input/plant-disease-classification-merged-dataset/Cassava__mosaic_disease
Folder 'Cassava__mosaic_disease' and its contents deleted.
Deleting: /kaggle/input/plant-disease-classification-merged-dataset/Cassava__brown_streak_disease
Folder 'Cassava__brown_streak_disease' and its contents deleted.
Deleting: /kaggle/input/plant-disease-classification-merged-dataset/Cassava__bacterial_blight
Folder 'Cassava__bacterial_blight' and its contents deleted.
Deleting: /kaggle/input/plant-disease-classification-merged-dataset/Corn__common_rust
Folder 'Corn__common_rust' and its contents deleted.
Deleting: /kaggle/input/plant-disease-classification-merged-dataset/Corn__northern_leaf_blight
Folder 'Corn_

In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split

# Set the path to your original dataset
original_dataset_dir = '/content/plant-disease-classification-merged-dataset'

# Create directories for training and testing data
base_dir = '/content/'
os.makedirs(base_dir, exist_ok=True)

train_dir = os.path.join(base_dir, 'train')
os.makedirs(train_dir, exist_ok=True)

test_dir = os.path.join(base_dir, 'test')
os.makedirs(test_dir, exist_ok=True)

# List all classes (subdirectories in the original dataset)
classes = os.listdir(original_dataset_dir)

# Set the ratio for training and testing
train_ratio = 0.8  # 80% for training, 20% for testing

# Iterate over each class
for class_name in classes:
    class_path = os.path.join(original_dataset_dir, class_name)

    # List all images in the class
    all_images = os.listdir(class_path)

    # Split the images into training and testing sets
    train_images, test_images = train_test_split(all_images, train_size=train_ratio, random_state=42)

    # Create directories for the class in the training and testing sets
    train_class_dir = os.path.join(train_dir, class_name)
    os.makedirs(train_class_dir, exist_ok=True)

    test_class_dir = os.path.join(test_dir, class_name)
    os.makedirs(test_class_dir, exist_ok=True)

    # Copy images to the training and testing directories
    for img_name in train_images:
        src = os.path.join(class_path, img_name)
        dst = os.path.join(train_class_dir, img_name)
        shutil.copyfile(src, dst)

    for img_name in test_images:
        src = os.path.join(class_path, img_name)
        dst = os.path.join(test_class_dir, img_name)
        shutil.copyfile(src, dst)

print('Dataset prepared.')

Dataset prepared.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, Input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Set the path to your dataset
train_data_dir = '/content/train'
test_data_dir = '/content/test'

# Define the image size and other parameters
img_width, img_height = 256, 256
batch_size = 128
epochs = 50  # Maximum number of epochs

# Create data generators for training and testing with additional augmentations
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True
)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_data_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_data_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical'
)

# Build the CNN model
model = models.Sequential()
model.add(Input(shape=(img_width, img_height, 3)))

model.add(layers.Conv2D(32, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))

model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))

model.add(layers.Conv2D(128, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))

model.add(layers.Flatten())
model.add(layers.Dense(256, activation='relu'))
model.add(layers.Dropout(0.5))
model.add(layers.Dense(len(train_generator.class_indices), activation='softmax'))

# Compile the model with a learning rate scheduler
initial_learning_rate = 0.001
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate,
    decay_steps=1000,
    decay_rate=0.96,
    staircase=True
)
optimizer = Adam(learning_rate=lr_schedule)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Define callbacks for early stopping and model checkpointing
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
model_checkpoint = ModelCheckpoint('/content/drive/MyDrive/plant_disease_model.h5', save_best_only=True)

# Train the model with callbacks
history = model.fit(
    train_generator,
    epochs=epochs,
    validation_data=test_generator,
    callbacks=[early_stopping, model_checkpoint]
)


Found 29106 images belonging to 50 classes.
Found 7300 images belonging to 50 classes.
Epoch 1/50
228/228 [==============================] - ETA: 0s - loss: 2.7471 - accuracy: 0.2763

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


228/228 [==============================] - 1015s 4s/step - loss: 2.7471 - accuracy: 0.2763 - val_loss: 1.7840 - val_accuracy: 0.4892
Epoch 2/50
228/228 [==============================] - 937s 4s/step - loss: 1.7279 - accuracy: 0.4915 - val_loss: 1.0714 - val_accuracy: 0.6667
Epoch 3/50
228/228 [==============================] - 927s 4s/step - loss: 1.4144 - accuracy: 0.5725 - val_loss: 1.0815 - val_accuracy: 0.6667
Epoch 4/50
228/228 [==============================] - 957s 4s/step - loss: 1.2282 - accuracy: 0.6233 - val_loss: 0.9752 - val_accuracy: 0.7068
Epoch 5/50
228/228 [==============================] - 960s 4s/step - loss: 1.1030 - accuracy: 0.6587 - val_loss: 0.9588 - val_accuracy: 0.7092
Epoch 6/50
228/228 [==============================] - 952s 4s/step - loss: 1.0075 - accuracy: 0.6898 - val_loss: 0.6650 - val_accuracy: 0.7899
Epoch 7/50
228/228 [==============================] - 918s 4s/step - loss: 0.9155 - accuracy: 0.7117 - val_loss: 0.7614 - val_accuracy: 0.7663
Epoch 8/5

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, Input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import os

# Set the path to your dataset
train_data_dir = '/content/train'
test_data_dir = '/content/test'

# Define the image size and other parameters
img_width, img_height = 256, 256
batch_size = 128
epochs = 50  # Maximum number of epochs

# Create data generators for training and testing with additional augmentations
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True
)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_data_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_data_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical'
)

# Build or load the CNN model
model_checkpoint_path = '/content/drive/MyDrive/plant_disease_model.h5'

if os.path.exists(model_checkpoint_path):
    # Load the saved model with the best weights
    model = tf.keras.models.load_model(model_checkpoint_path)
    # Find the number of epochs completed during the previous training session
    last_epoch = 41
    print("Resuming training from epoch", last_epoch + 1)
else:
    # Build the CNN model if no checkpoint exists
    model = models.Sequential()
    model.add(Input(shape=(img_width, img_height, 3)))
    model.add(layers.Conv2D(32, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Conv2D(64, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Conv2D(128, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Flatten())
    model.add(layers.Dense(256, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(len(train_generator.class_indices), activation='softmax'))

    # Compile the model
    initial_learning_rate = 0.001
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate,
        decay_steps=1000,
        decay_rate=0.96,
        staircase=True
    )
    optimizer = Adam(learning_rate=lr_schedule)
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Define callbacks for early stopping and model checkpointing
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
model_checkpoint = ModelCheckpoint(model_checkpoint_path, save_best_only=True)

# Train the model with callbacks
history = model.fit(
    train_generator,
    epochs=epochs,
    initial_epoch=last_epoch if 'last_epoch' in locals() else 0,
    validation_data=test_generator,
    callbacks=[early_stopping, model_checkpoint]
)


Found 29106 images belonging to 50 classes.
Found 7300 images belonging to 50 classes.
Resuming training from epoch 42
Epoch 42/50
228/228 [==============================] - ETA: 0s - loss: 0.5830 - accuracy: 0.8155

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


228/228 [==============================] - 966s 4s/step - loss: 0.5830 - accuracy: 0.8155 - val_loss: 0.3651 - val_accuracy: 0.8893
Epoch 43/50
228/228 [==============================] - 942s 4s/step - loss: 0.5628 - accuracy: 0.8209 - val_loss: 0.3317 - val_accuracy: 0.8951
Epoch 44/50
228/228 [==============================] - 941s 4s/step - loss: 0.5229 - accuracy: 0.8340 - val_loss: 0.3385 - val_accuracy: 0.8945
Epoch 45/50
228/228 [==============================] - 941s 4s/step - loss: 0.5044 - accuracy: 0.8399 - val_loss: 0.3234 - val_accuracy: 0.9004
Epoch 46/50
228/228 [==============================] - 920s 4s/step - loss: 0.5076 - accuracy: 0.8357 - val_loss: 0.4266 - val_accuracy: 0.8767
Epoch 47/50
228/228 [==============================] - 924s 4s/step - loss: 0.4933 - accuracy: 0.8424 - val_loss: 0.3376 - val_accuracy: 0.8932
Epoch 48/50
228/228 [==============================] - 969s 4s/step - loss: 0.4808 - accuracy: 0.8456 - val_loss: 0.3095 - val_accuracy: 0.9022
Epoc

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

# Set the path to your dataset
train_data_dir = '/content/train'

# Define the image size and other parameters
img_width, img_height = 256, 256
batch_size = 128

# Create data generator for training with augmentations
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True
)

# Create the train generator
train_generator = train_datagen.flow_from_directory(
    train_data_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical'
)

# Print the 50 classes present in the model
print("Classes present in the model:")
for class_name, class_index in train_generator.class_indices.items():
    print(f"Class {class_index}: {class_name}")

Found 29106 images belonging to 50 classes.
Classes present in the model:
Class 0: Apple__black_rot
Class 1: Apple__healthy
Class 2: Apple__rust
Class 3: Apple__scab
Class 4: Cherry__healthy
Class 5: Cherry__powdery_mildew
Class 6: Chili__healthy
Class 7: Chili__leaf curl
Class 8: Chili__leaf spot
Class 9: Chili__whitefly
Class 10: Chili__yellowish
Class 11: Coffee__cercospora_leaf_spot
Class 12: Coffee__healthy
Class 13: Coffee__red_spider_mite
Class 14: Coffee__rust
Class 15: Cucumber__diseased
Class 16: Cucumber__healthy
Class 17: Gauva__diseased
Class 18: Gauva__healthy
Class 19: Lemon__diseased
Class 20: Lemon__healthy
Class 21: Mango__diseased
Class 22: Mango__healthy
Class 23: Peach__bacterial_spot
Class 24: Peach__healthy
Class 25: Pepper_bell__bacterial_spot
Class 26: Pepper_bell__healthy
Class 27: Pomegranate__diseased
Class 28: Pomegranate__healthy
Class 29: Potato__early_blight
Class 30: Potato__healthy
Class 31: Potato__late_blight
Class 32: Strawberry___leaf_scorch
Class 